In [0]:
from pyspark.sql.functions import expr, current_timestamp
import unicodedata

In [0]:
# Caminho dos dados CSV dos voos
raw_path = '/Volumes/ct-esteira-dados-aviacao/raw/dados-aviacao-24'

In [0]:
# Leitura dos dados CSV dos voos
df_voos_raw_path = spark.read\
    .option("header", True) \
    .option("delimiter", ";") \
    .csv(raw_path)

In [0]:
def clean_col(col_name):
    col_name = col_name.strip().lower().replace(" ", "_")

    col_name = ''.join(
        c for c in unicodedata.normalize("NFKD", col_name)
        if not unicodedata.combining(c)
    )
    return col_name

df_voos_raw_path = df_voos_raw_path.toDF(
        *[clean_col(c) for c in df_voos_raw_path.columns]
    )


In [0]:
df_voos_raw_path.write \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .partitionBy("sigla_icao_empresa_aerea") \
    .saveAsTable("`ct-esteira-dados-aviacao`.raw.fat_voos_raw_dep_arr")